# What ARE the Most Metastable Nodes?

**Flipped question**: Instead of asking "are epileptic nodes metastable?", ask: 
"what ARE the most metastable nodes? What characterizes them? Do they have any clinical or anatomical meaning?"

Uses the Desikan-Killiany atlas column from implant Excel files to give anatomical labels.

Checks whether metastable nodes are:
1. At **probe boundaries** (first/last contacts on a depth electrode)
2. At **community boundaries** (high participation coefficient)
3. In specific **anatomical regions**
4. **Epileptic** nodes

In [ ]:
# Setup
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname='lrgeegfc')

import numpy as np
import pandas as pd
import re
import warnings
import openpyxl
from pathlib import Path
from collections import Counter
from scipy.cluster.hierarchy import fcluster

warnings.filterwarnings('ignore')

from lrg_eegfc.workflow.msc import load_msc_matrix
from lrg_eegfc.workflow.diagnostics import compute_coarsening_and_metastability
from lrg_eegfc.utils.io import load_epileptic_nodes

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────

def load_ch(patient):
    """Load channel labels, stripping spaces to match Excel format."""
    path = Path(f'data/stereoeeg_patients/{patient}/channel_labels.csv')
    with open(path) as f:
        first = f.readline().strip()
    if first.lower() == 'label':
        df = pd.read_csv(path, header=None, skiprows=1)
    else:
        df = pd.read_csv(path, header=None)
    return [str(l).strip('"').split(',')[0].strip().replace(' ', '') for l in df.iloc[:, 0]]


def get_probe(label):
    """Extract probe letter prefix from electrode label (e.g., 'A' from 'A3')."""
    m = re.match(r"([A-Za-z]+'?)", label)
    return m.group(1) if m else label


def load_implant_anatomy(patient):
    """Load anatomical labels from implant Excel file (Desikan-Killiany column)."""
    patnum = int(patient.split('_')[1])
    candidates = [
        Path(f'data/stereoeeg_patients/{patient}/Implant_pat_{patnum}.xlsx'),
        Path(f'data/stereoeeg_patients/{patient}/Implant_locations/Implant_pat_{patnum}.xlsx'),
    ]
    xlsx = next((c for c in candidates if c.exists()), None)
    if xlsx is None:
        return {}
    wb = openpyxl.load_workbook(xlsx, data_only=True)
    for name in ('all_leads', 'ALL_LEADS'):
        if name in wb.sheetnames:
            ws = wb[name]
            break
    else:
        ws = wb.active

    anatomy = {}
    for row in range(2, ws.max_row + 1):
        label = ws.cell(row, 1).value
        dk = ws.cell(row, 5).value  # Desikan-Killiany column
        if label and dk:
            clean_label = str(label).strip().replace(' ', '')
            anatomy[clean_label] = str(dk).strip()
    return anatomy

print("Helper functions defined.")

## Main Analysis: Per-patient, per-band metastability characterization

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
patients = ['Pat_02', 'Pat_03', 'Pat_05', 'Pat_07', 'Pat_08']
bands = ['delta', 'theta', 'alpha', 'beta']
TOP_K = 15  # number of top metastable nodes to inspect

# Collectors for cross-patient aggregation
all_rows = []  # one row per (patient, band, node-in-top-K)

for patient in patients:
    epi_set = set(load_epileptic_nodes(patient))
    ch = load_ch(patient)
    N = len(ch)
    epi_mask = np.array([l in epi_set for l in ch])
    anatomy = load_implant_anatomy(patient)

    print(f"\n{'=' * 80}")
    print(f"{patient}  ({N} channels, {sum(epi_mask)} epileptic, "
          f"{len(anatomy)} anatomy labels)")
    print(f"{'=' * 80}")

    for band in bands:
        A = load_msc_matrix(patient, 'rsPre', band,
                            cache_root=Path('data/msc_cache'),
                            sparsify='none', n_surrogates=0, nperseg=4096)
        if A is None or A.shape[0] != N:
            print(f"  {band}: SKIP (matrix mismatch or missing)")
            continue

        # Build Laplacian
        A_copy = A.copy()
        np.fill_diagonal(A_copy, 0)
        D = A_copy.sum(axis=1)
        L = np.diag(D) - A_copy
        evals, evecs = np.linalg.eigh(L)
        evals = np.maximum(evals, 0.0)
        if evals[1] < 1e-10:
            print(f"  {band}: SKIP (disconnected graph)")
            continue

        # Compute metastability
        result = compute_coarsening_and_metastability(evals, evecs, n_tau=40)
        mu = result['mu']

        # Top K most metastable
        top = np.argsort(mu)[::-1][:TOP_K]

        print(f"\n  rsPre/{band} -- Top {TOP_K} metastable nodes:")
        print(f"    {'#':>3s} {'label':>8s} {'probe':>6s} {'mu':>6s} "
              f"{'epi':>4s} {'strength':>8s} {'anatomy':>40s}")
        for rank, idx in enumerate(top):
            lbl = ch[idx]
            probe = get_probe(lbl)
            is_epi = 'EPI' if lbl in epi_set else ''
            strength = D[idx]
            anat = anatomy.get(lbl, '?')
            print(f"    {rank+1:3d} {lbl:>8s} {probe:>6s} {mu[idx]:6.3f} "
                  f"{is_epi:>4s} {strength:8.2f} {anat[:40]:>40s}")

            all_rows.append(dict(
                patient=patient, band=band, rank=rank + 1,
                label=lbl, probe=probe, mu=mu[idx],
                is_epi=lbl in epi_set, strength=strength,
                anatomy=anat,
            ))

        # ── Probe boundary analysis ──
        boundary_count = 0
        for idx in top:
            lbl = ch[idx]
            probe = get_probe(lbl)
            probe_contacts = sorted(
                [ch[i] for i in range(N) if get_probe(ch[i]) == probe])
            if lbl == probe_contacts[0] or lbl == probe_contacts[-1]:
                boundary_count += 1
        print(f"    Probe boundary contacts (first/last): {boundary_count}/{TOP_K}")

        # ── Probe distribution ──
        top_probes = [get_probe(ch[i]) for i in top]
        probe_counts = Counter(top_probes)
        print(f"    Probes represented: {dict(probe_counts)}")

        # ── Community boundary analysis (via LRG cache) ──
        lrg_path = Path(f'data/lrg_cache/{patient}/{band}_rsPre_lrg_msc.npz')
        if lrg_path.exists():
            lrg = np.load(lrg_path, allow_pickle=True)
            Z = lrg['linkage_matrix']
            thresh = float(lrg['optimal_threshold'])
            n_nodes_lrg = int(lrg['n_nodes'])
            if n_nodes_lrg == N:
                labels_opt = fcluster(Z, thresh, criterion='distance')
                n_comm = len(np.unique(labels_opt))

                # Participation coefficient
                pc = np.zeros(N)
                for i in range(N):
                    si = A_copy[i].sum()
                    if si == 0:
                        continue
                    for c in np.unique(labels_opt):
                        pc[i] += (A_copy[i, labels_opt == c].sum() / si) ** 2
                pc = 1 - pc

                top_mask = np.zeros(N, dtype=bool)
                top_mask[top] = True

                # Community boundary nodes: have strong neighbors in >1 community
                boundary_nodes = 0
                for idx in top:
                    nonzero = A_copy[idx][A_copy[idx] > 0]
                    if len(nonzero) == 0:
                        continue
                    thresh50 = np.percentile(nonzero, 50)
                    neighbor_comms = set(
                        labels_opt[A_copy[idx] > thresh50])
                    if len(neighbor_comms) > 1:
                        boundary_nodes += 1

                print(f"    Community boundary nodes: {boundary_nodes}/{TOP_K} "
                      f"(n*={n_comm})")
                print(f"    Participation coeff: top_meta={pc[top_mask].mean():.3f} "
                      f"vs others={pc[~top_mask].mean():.3f}")

print("\n\nPer-patient analysis complete.")

## Cross-patient aggregate statistics

Now pool all top-metastable nodes across patients and bands to find patterns.

In [ ]:
df = pd.DataFrame(all_rows)
print(f"Total rows: {len(df)}")
print(f"Patients: {df['patient'].nunique()}")
print(f"Bands: {df['band'].unique().tolist()}")

# ── 1. Epileptic fraction among top metastable nodes ──
n_epi = df['is_epi'].sum()
n_total = len(df)
print(f"\n--- Epileptic nodes among top-{TOP_K} metastable ---")
print(f"  {n_epi}/{n_total} = {100 * n_epi / n_total:.1f}%")

# Compare to baseline: what fraction of ALL nodes are epileptic?
baseline_epi_fracs = []
for patient in patients:
    epi_set = set(load_epileptic_nodes(patient))
    ch = load_ch(patient)
    frac = sum(1 for l in ch if l in epi_set) / len(ch)
    baseline_epi_fracs.append(frac)
    print(f"  {patient} baseline epileptic fraction: {100 * frac:.1f}%")
print(f"  Mean baseline: {100 * np.mean(baseline_epi_fracs):.1f}%")

# ── 2. Anatomy distribution ──
print(f"\n--- Anatomical regions (Desikan-Killiany) in top metastable ---")
anat_counts = df[df['anatomy'] != '?']['anatomy'].value_counts()
print(anat_counts.head(20).to_string())
print(f"\n  Unique regions: {df[df['anatomy'] != '?']['anatomy'].nunique()}")
print(f"  Missing anatomy ('?'): {(df['anatomy'] == '?').sum()}/{n_total}")

# ── 3. Probe boundary fraction (need to recompute) ──
print(f"\n--- Probe boundary analysis ---")
boundary_counts_per_band = {}
for band in bands:
    sub = df[df['band'] == band]
    # We stored labels and probes; check if label is first or last on its probe
    # within each patient
    boundary = 0
    for _, row in sub.iterrows():
        ch = load_ch(row['patient'])
        probe = row['probe']
        probe_contacts = sorted([c for c in ch if get_probe(c) == probe])
        if row['label'] == probe_contacts[0] or row['label'] == probe_contacts[-1]:
            boundary += 1
    boundary_counts_per_band[band] = (boundary, len(sub))
    print(f"  {band}: {boundary}/{len(sub)} = {100 * boundary / len(sub):.1f}% "
          f"are probe boundary contacts")

total_boundary = sum(v[0] for v in boundary_counts_per_band.values())
total_n = sum(v[1] for v in boundary_counts_per_band.values())
print(f"  TOTAL: {total_boundary}/{total_n} = {100 * total_boundary / total_n:.1f}%")

# ── 4. Per-band epileptic breakdown ──
print(f"\n--- Epileptic fraction per band ---")
for band in bands:
    sub = df[df['band'] == band]
    n_e = sub['is_epi'].sum()
    print(f"  {band}: {n_e}/{len(sub)} = {100 * n_e / len(sub):.1f}%")

## Parse dominant cortical region from probabilistic anatomy strings

The Excel column 5 contains probabilistic tissue assignments like
`ctx-lh-superiorfrontal,66.0,Wm,22.0,...`. We extract the **dominant cortical region** (ignoring Wm/Unk) for meaningful anatomical grouping.

In [ ]:
def parse_dominant_region(anat_string):
    """Extract the dominant cortical region from a probabilistic anatomy string.
    
    E.g. 'ctx-lh-superiorfrontal,66.0,Wm,22.0,Unk,12.0,PTD, 0.56'
    -> 'superiorfrontal (L)' (the dominant cortical region)
    
    Returns 'white_matter' if Wm dominates, 'unknown' if Unk dominates,
    or the cortical/subcortical label.
    """
    if not anat_string or anat_string == '?':
        return 'no_data'
    
    # Parse pairs: (tissue_label, percentage)
    parts = anat_string.replace('PTD,', '').strip().split(',')
    pairs = []
    i = 0
    while i < len(parts) - 1:
        label = parts[i].strip()
        try:
            pct = float(parts[i + 1].strip())
            pairs.append((label, pct))
            i += 2
        except ValueError:
            i += 1
    
    if not pairs:
        return 'unparseable'
    
    cortical = [(lbl, pct) for lbl, pct in pairs if lbl.startswith('ctx-')]
    subcortical = [(lbl, pct) for lbl, pct in pairs 
                   if not lbl.startswith('ctx-') and lbl not in ('Wm', 'Unk')]
    wm_pct = sum(pct for lbl, pct in pairs if lbl == 'Wm')
    unk_pct = sum(pct for lbl, pct in pairs if lbl == 'Unk')
    
    if cortical:
        best = max(cortical, key=lambda x: x[1])
        short = best[0].split('-', 2)[-1] if '-' in best[0] else best[0]
        hemi = 'L' if '-lh-' in best[0] else ('R' if '-rh-' in best[0] else '')
        return f"{short} ({hemi})" if hemi else short
    elif subcortical:
        best = max(subcortical, key=lambda x: x[1])
        return best[0]
    elif wm_pct >= unk_pct:
        return 'white_matter'
    else:
        return 'unknown'


def broad_category(region):
    """Map DK regions to broad lobar categories."""
    if region in ('white_matter', 'unknown', 'no_data', 'unparseable'):
        return region
    frontal = ['superiorfrontal', 'caudalmiddlefrontal', 'rostralmiddlefrontal',
               'parsopercularis', 'parsorbitalis', 'parstriangularis',
               'lateralorbitofrontal', 'medialorbitofrontal', 'frontalpole',
               'caudalanteriorcingulate', 'rostralanteriorcingulate', 'precentral']
    temporal = ['superiortemporal', 'middletemporal', 'inferiortemporal',
                'transversetemporal', 'bankssts', 'fusiform', 'entorhinal',
                'temporalpole', 'parahippocampal']
    parietal = ['superiorparietal', 'inferiorparietal', 'supramarginal',
                'postcentral', 'precuneus']
    occipital = ['lateraloccipital', 'lingual', 'cuneus', 'pericalcarine']
    limbic = ['posteriorcingulate', 'isthmuscingulate', 'insula']
    
    base = region.split(' (')[0]
    if base in frontal:
        return 'frontal'
    elif base in temporal:
        return 'temporal'
    elif base in parietal:
        return 'parietal'
    elif base in occipital:
        return 'occipital'
    elif base in limbic:
        return 'limbic'
    elif 'Cerebellum' in region:
        return 'cerebellum'
    else:
        return 'other'


# Apply to the dataframe
df['region'] = df['anatomy'].apply(parse_dominant_region)
df['broad_region'] = df['region'].apply(broad_category)

print("Dominant cortical region distribution in top-metastable nodes:\n")
region_counts = df['region'].value_counts()
print(region_counts.to_string())
print(f"\nTotal unique regions: {df['region'].nunique()}")

print("\n\nBroad region distribution:")
broad_counts = df['broad_region'].value_counts()
print(broad_counts.to_string())

# Per-patient enrichment
print("\n\nPer-patient: broad region enrichment (meta vs implant)")
for patient in patients:
    anatomy = load_implant_anatomy(patient)
    ch_labels = load_ch(patient)
    
    all_regions = [broad_category(parse_dominant_region(anatomy.get(l, '?'))) 
                   for l in ch_labels]
    all_counts = Counter(all_regions)
    total_all = len(ch_labels)
    
    sub = df[df['patient'] == patient]
    meta_counts = Counter(sub['broad_region'].values)
    total_meta = len(sub)
    
    print(f"\n  {patient} ({total_all} ch, {total_meta} meta entries)")
    print(f"    {'Category':<15s} {'Implant%':>9s} {'Meta%':>9s} {'Ratio':>7s}")
    all_cats = sorted(set(list(all_counts.keys()) + list(meta_counts.keys())))
    for cat in all_cats:
        a_frac = all_counts.get(cat, 0) / total_all
        m_frac = meta_counts.get(cat, 0) / total_meta
        ratio = m_frac / a_frac if a_frac > 0 else float('inf')
        marker = ' **' if ratio > 1.5 else (' --' if ratio < 0.5 else '')
        print(f"    {cat:<15s} {100*a_frac:8.1f}% {100*m_frac:8.1f}% "
              f"{ratio:6.2f}x{marker}")

## Anatomical enrichment analysis

Are certain brain regions over-represented among metastable nodes compared to the implant as a whole?

In [ ]:
# For each patient, compare anatomical distribution in top-metastable vs ALL nodes
print("=== Anatomical enrichment: top-metastable vs whole implant ===\n")

for patient in patients:
    anatomy = load_implant_anatomy(patient)
    ch = load_ch(patient)
    
    # All-node anatomy distribution
    all_anat = [anatomy.get(l, '?') for l in ch]
    all_anat_counts = Counter(a for a in all_anat if a != '?')
    total_with_anat = sum(all_anat_counts.values())
    
    if total_with_anat == 0:
        print(f"{patient}: no anatomy data available\n")
        continue
    
    # Top-metastable anatomy (across all bands)
    sub = df[(df['patient'] == patient) & (df['anatomy'] != '?')]
    if len(sub) == 0:
        print(f"{patient}: no anatomy matches for top metastable nodes\n")
        continue
    
    meta_anat_counts = Counter(sub['anatomy'].values)
    
    print(f"{patient}  (total channels with anatomy: {total_with_anat})")
    print(f"  {'Region':<35s} {'All%':>6s} {'Meta%':>6s} {'Enrichment':>10s}")
    
    # Get all regions that appear in either
    all_regions = set(all_anat_counts.keys()) | set(meta_anat_counts.keys())
    enrichment_data = []
    for region in all_regions:
        all_frac = all_anat_counts.get(region, 0) / total_with_anat
        meta_frac = meta_anat_counts.get(region, 0) / len(sub)
        enrichment = meta_frac / all_frac if all_frac > 0 else float('inf')
        enrichment_data.append((region, all_frac, meta_frac, enrichment))
    
    # Sort by enrichment
    enrichment_data.sort(key=lambda x: x[3], reverse=True)
    for region, all_frac, meta_frac, enrichment in enrichment_data:
        if meta_frac > 0:  # only show regions present in metastable set
            marker = ' ***' if enrichment > 2.0 else (' **' if enrichment > 1.5 else '')
            print(f"  {region:<35s} {100*all_frac:5.1f}% {100*meta_frac:5.1f}% "
                  f"{enrichment:9.2f}x{marker}")
    
    # Regions in implant but NOT in metastable set
    depleted = [r for r, af, mf, e in enrichment_data if mf == 0 and af > 0.03]
    if depleted:
        print(f"  Depleted (>3% of implant, 0% in meta): {depleted}")
    print()

## Node strength vs metastability

Are metastable nodes hubs (high strength) or peripheral (low strength)?

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(patients), len(bands), figsize=(16, 4 * len(patients)),
                          squeeze=False)
fig.suptitle('Node strength vs metastability (mu)', fontsize=14, y=1.01)

for pi, patient in enumerate(patients):
    epi_set = set(load_epileptic_nodes(patient))
    ch = load_ch(patient)
    N = len(ch)
    epi_mask = np.array([l in epi_set for l in ch])
    
    for bi, band in enumerate(bands):
        ax = axes[pi, bi]
        
        A = load_msc_matrix(patient, 'rsPre', band,
                            cache_root=Path('data/msc_cache'),
                            sparsify='none', n_surrogates=0, nperseg=4096)
        if A is None or A.shape[0] != N:
            ax.set_visible(False)
            continue
        
        A_copy = A.copy()
        np.fill_diagonal(A_copy, 0)
        D = A_copy.sum(axis=1)
        L = np.diag(D) - A_copy
        evals, evecs = np.linalg.eigh(L)
        evals = np.maximum(evals, 0.0)
        if evals[1] < 1e-10:
            ax.set_visible(False)
            continue
        
        result = compute_coarsening_and_metastability(evals, evecs, n_tau=40)
        mu = result['mu']
        
        # Scatter: non-epileptic in blue, epileptic in red
        ax.scatter(D[~epi_mask], mu[~epi_mask], alpha=0.4, s=15, c='steelblue',
                   label='non-EPI')
        ax.scatter(D[epi_mask], mu[epi_mask], alpha=0.7, s=25, c='red',
                   marker='^', label='EPI')
        
        # Correlation
        corr = np.corrcoef(D, mu)[0, 1]
        ax.set_title(f'{patient} {band}\nr={corr:.2f}', fontsize=9)
        ax.set_xlabel('Strength' if pi == len(patients) - 1 else '')
        ax.set_ylabel('mu' if bi == 0 else '')
        if pi == 0 and bi == 0:
            ax.legend(fontsize=7)

fig.tight_layout()
plt.show()
plt.close(fig)

## Band consistency: are the same nodes metastable across frequency bands?

In [ ]:
print("=== Band consistency: how often does a node appear in top-15 across bands? ===\n")

for patient in patients:
    sub = df[df['patient'] == patient]
    label_band_counts = sub.groupby('label')['band'].nunique()
    
    # Nodes appearing in top-15 for multiple bands
    multi_band = label_band_counts[label_band_counts > 1]
    
    print(f"{patient}:")
    print(f"  Unique nodes in top-15 across all bands: {sub['label'].nunique()}")
    print(f"  Nodes in top-15 for >=2 bands: {len(multi_band)}")
    if len(multi_band) > 0:
        for label, count in multi_band.sort_values(ascending=False).items():
            bands_for_label = sub[sub['label'] == label]['band'].tolist()
            anat = sub[sub['label'] == label]['anatomy'].iloc[0]
            is_epi = sub[sub['label'] == label]['is_epi'].iloc[0]
            epi_str = ' [EPI]' if is_epi else ''
            print(f"    {label}: {count} bands ({', '.join(bands_for_label)}) "
                  f"— {anat}{epi_str}")
    print()

## Summary table

In [ ]:
# Final summary table
print("=" * 80)
print("SUMMARY: What characterizes the most metastable nodes?")
print("=" * 80)

# 1. Epileptic
n_epi_meta = df['is_epi'].sum()
n_total_meta = len(df)
print(f"\n1. EPILEPTIC STATUS")
print(f"   {n_epi_meta}/{n_total_meta} ({100*n_epi_meta/n_total_meta:.1f}%) of top-{TOP_K} "
      f"metastable nodes are epileptic")
print(f"   Baseline epileptic fraction: ~{100*np.mean(baseline_epi_fracs):.1f}%")

# 2. Anatomy
print(f"\n2. ANATOMICAL REGIONS (most common in top metastable)")
known = df[df['anatomy'] != '?']
if len(known) > 0:
    top_regions = known['anatomy'].value_counts().head(10)
    for region, count in top_regions.items():
        print(f"   {region}: {count} ({100*count/len(known):.1f}%)")

# 3. Strength
print(f"\n3. NODE STRENGTH")
print(f"   Mean strength of top-metastable: {df['strength'].mean():.2f}")

# 4. Probe boundaries
print(f"\n4. PROBE BOUNDARIES")
print(f"   {total_boundary}/{total_n} ({100*total_boundary/total_n:.1f}%) of top-metastable "
      f"are first/last contacts on their probe")

# 5. Band consistency
multi_band_total = 0
unique_total = 0
for patient in patients:
    sub = df[df['patient'] == patient]
    label_band_counts = sub.groupby('label')['band'].nunique()
    multi_band_total += (label_band_counts > 1).sum()
    unique_total += len(label_band_counts)
print(f"\n5. BAND CONSISTENCY")
print(f"   {multi_band_total}/{unique_total} ({100*multi_band_total/unique_total:.1f}%) of "
      f"unique metastable nodes appear in top-{TOP_K} for multiple bands")